In [18]:
import torch
from torch.utils.data import IterableDataset, DataLoader
import pandas as pd
import numpy as np
from pathlib import Path
import random
import pyarrow.parquet as pq


class FastDataset(IterableDataset):
    def __init__(self, folder_list):
        self.folder_list = folder_list
        self.union_feature_files = [
            ("feat1.parquet", [f'feat1_{j}' for j in range(10)]),
            ("feat2.parquet", [f'feat2_{j}' for j in range(10)]),
        ]
        self.separate_file_prefix = "large_data_part"
        self.separate_file_columns = [f'col_{k}' for k in range(100)]
        self.max_parts = 10
        self.dataset_len = 0
        for folder in self.folder_list:
            folder_path = Path(folder)
            parquet_file = pq.ParquetFile(folder_path / self.union_feature_files[0][0])
            total_rows = parquet_file.metadata.num_rows
            self.dataset_len += total_rows

    def reset(self):
        random.shuffle(self.folder_list)

    def __len__(self):
        return self.dataset_len

    def __iter__(self):
        worker_info = torch.utils.data.get_worker_info()
        worker_id, num_workers = (worker_info.id, worker_info.num_workers) if worker_info else (0, 1)
        for folder in self.folder_list:
            folder_path = Path(folder)
            union_index = 0
            parquet_file = pq.ParquetFile(folder_path / self.union_feature_files[0][0])
            total_rows = parquet_file.metadata.num_rows
            if num_workers > 1:
                union_left = total_rows * worker_id // num_workers
                union_right = total_rows * (worker_id + 1) // num_workers
            else:
                union_left = 0
                union_right = total_rows

            for part in range(self.max_parts):
                separate_file = folder_path / f"{self.separate_file_prefix}{part}.parquet"
                if not separate_file.exists():
                    break
                parquet_file = pq.ParquetFile(separate_file)
                n_rows = parquet_file.metadata.num_rows
                union_index_new = union_index + n_rows
                if union_index_new <= union_left:
                    union_index = union_index_new
                    continue
                separate_df = parquet_file.read(columns=self.separate_file_columns).to_pandas()
                skipped_rows = max(0, union_left - union_index)
                omitted_rows = max(0, union_index_new - union_right)
                union_dfs = []
                
                for union_file, columns in self.union_feature_files:
                    union_file_path = folder_path / union_file
                    df = pd.read_parquet(union_file_path, columns=columns)
                    union_dfs.append(df.iloc[union_index + skipped_rows:union_index_new - omitted_rows].reset_index(drop=True))
                union_df = pd.concat(union_dfs, axis=1)
                df = pd.concat([separate_df.iloc[skipped_rows:-omitted_rows].reset_index(drop=True), union_df], axis=1)
                union_index = union_index_new
                for _, row in df.iterrows():
                    yield torch.tensor(row.values, dtype=torch.float32)
                if union_index >= union_right:
                    break
            
            assert union_index == union_right, f"union_index: {union_index}, union_right: {union_right}"

In [20]:
import datetime
import pandas as pd 
import numpy as np 
from pathlib import Path
import time

start_date = datetime.date(2025, 1, 1)
end_date = datetime.date(2025, 1, 31)

folder_list = []
for i in range((end_date - start_date).days + 1):
    current_date = start_date + datetime.timedelta(days=i)
    folder = Path(f"example/data_{current_date}")
    folder_list.append(str(folder))

dataset = FastDataset(folder_list)
print(f"Dataset length: {len(dataset)}")
# shuffling is achieved by shuffling folder_list
dataloader = DataLoader(dataset, batch_size=64, shuffle=False, num_workers=4, pin_memory=False, prefetch_factor=2, persistent_workers=False, multiprocessing_context="spawn")
print(f"len(dataloader): {len(dataloader)} batches")

for epoch in range(5):
    t1 = time.time()
    dataset.reset()
    t2 = time.time()
    print(f"Epoch {epoch}, reset time: {t2 - t1:.6f} seconds")
    for batch_idx, batch in enumerate(dataloader):
        pass
    t3 = time.time()
    print(f"Epoch {epoch}, data loading time: {t3 - t2:.6f} seconds")


Dataset length: 310000
len(dataloader): 4844 batches
Epoch 0, reset time: 0.000023 seconds


Traceback (most recent call last):
Traceback (most recent call last):
  File "<string>", line 1, in <module>
Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/Users/runtianzhai/mambaforge/envs/ml/lib/python3.11/multiprocessing/spawn.py", line 122, in spawn_main
Traceback (most recent call last):
  File "/Users/runtianzhai/mambaforge/envs/ml/lib/python3.11/multiprocessing/spawn.py", line 122, in spawn_main
  File "<string>", line 1, in <module>
  File "<string>", line 1, in <module>
  File "/Users/runtianzhai/mambaforge/envs/ml/lib/python3.11/multiprocessing/spawn.py", line 122, in spawn_main
  File "/Users/runtianzhai/mambaforge/envs/ml/lib/python3.11/multiprocessing/spawn.py", line 122, in spawn_main
    exitcode = _main(fd, parent_sentinel)
    exitcode = _main(fd, parent_sentinel)
    exitcode = _main(fd, parent_sentinel)
    exitcode = _main(fd, parent_sentinel)  
                      ^ ^ ^ ^  ^ ^ ^  ^   ^  ^ ^ ^       ^ ^ ^       ^  ^ ^^^^ ^^^^^^^

RuntimeError: DataLoader worker (pid(s) 59619) exited unexpectedly

# Build example dataset

In [27]:
import datetime
import pandas as pd 
import numpy as np 
from pathlib import Path

start_date = datetime.date(2025, 1, 1)
end_date = datetime.date(2025, 1, 31)

for i in range((end_date - start_date).days + 1):
    current_date = start_date + datetime.timedelta(days=i)
    # Create a dataframe with 10,000 rows and 10 columns of random values
    folder = Path(f"example/data_{current_date}")
    folder.mkdir(parents=True, exist_ok=True)
    df1 = pd.DataFrame(np.random.rand(9997, 10), columns=[f'feat1_{j}' for j in range(10)])
    df1.to_parquet(f'example/data_{current_date}/feat1.parquet')
    df2 = pd.DataFrame(np.random.rand(9997, 10), columns=[f'feat2_{j}' for j in range(10)])
    df2.to_parquet(f'example/data_{current_date}/feat2.parquet')

In [28]:
for i in range((end_date - start_date).days + 1):
    current_date = start_date + datetime.timedelta(days=i)
    sizes = [1000, 1500, 2000, 2500, 2997]
    for j in range(5):
        size = sizes[j]
        df = pd.DataFrame(np.random.rand(size, 100), columns=[f'col_{k}' for k in range(100)])
        df.to_parquet(f'example/data_{current_date}/large_data_part{j}.parquet')

In [29]:
df = pd.read_parquet("/Users/runtianzhai/Documents/project/Python/kaggle/example/data_2025-01-01/feat1.parquet")
df

,feat1_0,feat1_1,feat1_2,feat1_3,feat1_4,feat1_5,feat1_6,feat1_7,feat1_8,feat1_9
0,0.679862,0.726828,0.196228,0.136774,0.817699,0.592953,0.370978,0.398121,0.238058,0.882915
1,0.362774,0.981556,0.099991,0.806795,0.746477,0.345990,0.110738,0.982277,0.164091,0.168689
2,0.839598,0.171912,0.929826,0.236857,0.716672,0.075594,0.863552,0.192675,0.307698,0.931940
3,0.715600,0.738976,0.831793,0.795844,0.962719,0.794982,0.074072,0.586019,0.698264,0.793799
4,0.215654,0.590576,0.767526,0.406814,0.839939,0.661863,0.551225,0.220953,0.561727,0.893086
...,...,...,...,...,...,...,...,...,...,...
9992,0.412963,0.863360,0.462918,0.100512,0.367667,0.949657,0.690088,0.601484,0.332061,0.986557
9993,0.715137,0.537276,0.338896,0.005280,0.063887,0.968672,0.600178,0.227664,0.814971,0.838606
9994,0.451533,0.639244,0.491576,0.180352,0.521314,0.340703,0.364416,0.957297,0.516716,0.458233
9995,0.675162,0.122646,0.915480,0.145896,0.937113,0.665626,0.698034,0.629445,0.905537,0.525620
